# Bronze ingest — Auto Loader availableNow

Reads new CSV files from ADLS via Auto Loader (`cloudFiles`), appends to managed Delta bronze tables, then stops.
Checkpoints under `autoloader_state_path` ensure each file is processed once.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "ingestion")
dbutils.widgets.text(
    "landing_path",
    "abfss://metastore@dbxucfc9c48d2meta.dfs.core.windows.net/actuarial/ingestion/landing",
)
dbutils.widgets.text(
    "autoloader_state_path",
    "abfss://metastore@dbxucfc9c48d2meta.dfs.core.windows.net/actuarial/ingestion/_autoloader",
)

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
autoloader_state_path = dbutils.widgets.get("autoloader_state_path").rstrip("/")

print(f"catalog={catalog}  schema={schema}")
print(f"landing_path={landing_path}")
print(f"autoloader_state_path={autoloader_state_path}")

In [ ]:
from actuarial_ingestion_adls.auto_loader import DATASETS, ingest_all_datasets

tables = ingest_all_datasets(
    spark,
    catalog=catalog,
    schema=schema,
    landing_path=landing_path,
    autoloader_state_path=autoloader_state_path,
    datasets=DATASETS,
)

print(f"{'─'*60}")
print(f"Bronze ingest complete — {len(tables)} table(s) in {catalog}.{schema}")
print(f"{'─'*60}")
for table in tables:
    count = spark.table(table).count()
    print(f"  {table:55s}  ({count:,} rows)")